In [1]:
from warnings import filterwarnings
filterwarnings("ignore")

In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import QuantileTransformer
from sklearn.linear_model import LogisticRegression
np.random.seed(2017)

In [3]:
FEATURES = ['score' + str(x+1) for x in range(3)]
TARGET = ['target']

In [4]:
train = pd.read_csv('../../data/combined/data.csv')
train = train[train['source'] == 'ISIC20']
train = train[['image_id','fold','target']]
score1 = pd.read_csv('../../score/regress_0_train.csv')[['image_id','score']]
score1.columns = ['image_id','score1']
score2 = pd.read_csv('../../score/regress_1_train.csv')[['image_id','score']]
score2.columns = ['image_id','score2']
score3 = pd.read_csv('../../score/regress_2_train.csv')[['image_id','score']]
score3.columns = ['image_id','score3']
train = train.merge(score1, on='image_id')
train = train.merge(score2, on='image_id')
train = train.merge(score3, on='image_id')

In [5]:
for idx in range(3):
    idx += 1
    try:
        print('Model:',idx,'AUC:',roc_auc_score(train.target, train['score'+str(idx)]))
    except:
        pass

Model: 1 AUC: 0.9427017497404309
Model: 2 AUC: 0.9495862705169569
Model: 3 AUC: 0.9381094248417793


In [6]:
test = pd.read_csv('../../data/combined/test.csv')
test = test.rename(columns={'image_name':'image_id'})[['image_id']]
score1 = pd.read_csv('../../score/regress_0_test.csv')[['image_name','target']]
score1.columns = ['image_id','score1']
score2 = pd.read_csv('../../score/regress_1_test.csv')[['image_name','target']]
score2.columns = ['image_id','score2']
score3 = pd.read_csv('../../score/regress_2_test.csv')[['image_name','target']]
score3.columns = ['image_id','score3']
test = test.merge(score1, on='image_id')
test = test.merge(score2, on='image_id')
test = test.merge(score3, on='image_id')

In [7]:
train.head().round(3)

,image_id,fold,target,score1,score2,score3
0,ISIC_2637011,0,0,0.002,0.000,0.004
1,ISIC_0015719,2,0,0.000,0.000,0.000
2,ISIC_0052212,4,0,0.001,0.000,0.000
3,ISIC_0068279,0,0,0.002,0.007,0.006
4,ISIC_0074268,4,0,0.000,0.000,0.000


In [8]:
test.head().round(3)

,image_id,score1,score2,score3
0,ISIC_0052060,0.001,0.000,0.001
1,ISIC_0052349,0.000,0.000,0.001
2,ISIC_0058510,0.000,0.000,0.000
3,ISIC_0073313,0.000,0.000,0.000
4,ISIC_0073502,0.005,0.004,0.037


In [9]:
train.describe().round(2).T

,count,mean,std,min,25%,50%,75%,max
fold,32691.0,2.02,1.42,0.0,1.0,2.0,3.00,4.00
target,32691.0,0.02,0.13,0.0,0.0,0.0,0.00,1.00
score1,32691.0,0.02,0.06,0.0,0.0,0.0,0.01,0.98
score2,32691.0,0.02,0.07,0.0,0.0,0.0,0.01,0.99
score3,32691.0,0.02,0.06,0.0,0.0,0.0,0.01,0.99


In [10]:
test.describe().round(2).T

,count,mean,std,min,25%,50%,75%,max
score1,10982.0,0.02,0.07,0.0,0.0,0.0,0.01,0.94
score2,10982.0,0.02,0.07,0.0,0.0,0.0,0.01,0.92
score3,10982.0,0.03,0.08,0.0,0.0,0.0,0.02,0.98


In [11]:
train[[x for x in FEATURES]].corr().round(4)

,score1,score2,score3
score1,1.0000,0.8553,0.8213
score2,0.8553,1.0000,0.8587
score3,0.8213,0.8587,1.0000


In [12]:
test[[x for x in FEATURES]].corr().round(4)

,score1,score2,score3
score1,1.0000,0.8839,0.8180
score2,0.8839,1.0000,0.8799
score3,0.8180,0.8799,1.0000


In [13]:
def fitModel(fold):
    scale = QuantileTransformer(n_quantiles=100, output_distribution='normal')
    train_subset = train[train['fold'] != fold].reset_index(drop=True).copy()
    valid_subset = train[train['fold'] == fold].reset_index(drop=True).copy()
    print('Data:', train_subset.shape, valid_subset.shape)
    valid_driver = valid_subset[['image_id','target']].copy()
    test_driver = test[['image_id']].copy()
    X_train, y_train = train_subset[FEATURES], train_subset[TARGET]
    X_valid, y_valid = valid_subset[FEATURES], valid_subset[TARGET]
    X_test = test[FEATURES]
    X_train = scale.fit_transform(X_train)
    X_valid = scale.transform(X_valid)
    X_test = scale.transform(X_test)
    model = LogisticRegression(C=1., fit_intercept=True)
    model.fit(X_train, y_train)
    print('Weights:', model.coef_.round(4))
    valid_driver['score'] = model.predict_proba(X_valid)[:,1]
    test_driver['score'] = model.predict_proba(X_test)[:,1]
    print('Stacked AUC:', round(roc_auc_score(valid_driver.target, valid_driver.score), 4))
    print('Output:', test_driver.shape)
    return valid_driver, test_driver

In [14]:
valid_0, test_0 = fitModel(0)

Data: (26213, 6) (6478, 6)
Weights: [[ 1.1411  1.7591 -0.0267]]
Stacked AUC: 0.9597
Output: (10982, 2)


In [15]:
valid_1, test_1 = fitModel(1)

Data: (26217, 6) (6474, 6)
Weights: [[1.04   1.5957 0.3269]]
Stacked AUC: 0.9407
Output: (10982, 2)


In [16]:
valid_2, test_2 = fitModel(2)

Data: (26263, 6) (6428, 6)
Weights: [[1.0221 1.789  0.0958]]
Stacked AUC: 0.9439
Output: (10982, 2)


In [17]:
valid_3, test_3 = fitModel(3)

Data: (26013, 6) (6678, 6)
Weights: [[1.1144 1.6722 0.1162]]
Stacked AUC: 0.9524
Output: (10982, 2)


In [18]:
valid_4, test_4 = fitModel(4)

Data: (26058, 6) (6633, 6)
Weights: [[0.9222 1.6689 0.2706]]
Stacked AUC: 0.9614
Output: (10982, 2)


In [19]:
valid_data = valid_0.append(valid_1).append(valid_2).append(valid_3).append(valid_4)

In [20]:
test_data = test_0.append(test_1).append(test_2).append(test_3).append(test_4)

In [21]:
valid_data = valid_data.groupby(['image_id','target']).mean().reset_index()
test_data = test_data.groupby('image_id').mean().reset_index()

In [22]:
valid_data.shape, test_data.shape

((32691, 3), (10982, 2))

In [23]:
roc_auc_score(valid_data.target, valid_data.score)

0.9517311136256553

In [24]:
test_data.columns = ['image_name','target']
test_data.to_csv('../../score/regress.csv', index=False)

In [25]:
# Metadata Y: 0.9548
# Metadata N: 0.9548